# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)



Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


In [4]:
MONTHS = [
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

monthly_queries = []

for month in MONTHS:
    monthly_queries.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            '{month}' AS month_key,

            COUNT(*) AS observed_days,

            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position,

            AVG(gsc_impressions) AS daily_impression_mean,
            STDDEV_SAMP(gsc_impressions) AS daily_impression_std,

            STDDEV_SAMP(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS daily_position_std,

            SUM(
                CASE
                    WHEN gsc_impressions > 0 THEN 1
                    ELSE 0
                END
            ) AS days_with_impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/'
            'month={month}/*.parquet'
        )

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    """)

monthly_union_sql = "\nUNION ALL\n".join(monthly_queries)

monthly_summary = con.sql(monthly_union_sql).df()

monthly_summary = monthly_summary.sort_values(
    ["month_key", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

print(f"Monthly summary rows: {len(monthly_summary):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly summary rows: 971,603


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.